# Correction effects across all MOUSE configurations

> **Supplementary poster/testing notebook.** Original processing date: **2026-09-15**; MoDaCor version: **1.8.0**. This is not part of the core MOUSE correction example.

This notebook extends the configuration-specific poster comparison to all ten measurements in sample batch 2. The upper and lower axes share the same logarithmic $q$ axis with a 3:1 height ratio.

For every configuration, the raw detector counts receive no intensity corrections but are integrated with that configuration's final mask, geometry, and bins. A separate robust global scale aligns each raw integration with its corrected counterpart; the remaining ordinary-percentage difference is shown on the lower symlog axis.

In [ ]:
from pathlib import Path
import json
import logging
import sys

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D
import numpy as np

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "example_utils.py").is_file():
        EXAMPLES_ROOT = candidate
        break
else:
    raise FileNotFoundError("Start Jupyter from MoDaCor_examples or one of its subdirectories.")

sys.path.insert(0, str(EXAMPLES_ROOT))
from example_utils import locate_example_dir
from modacor.dataclasses.processing_data import ProcessingData
from modacor.io.hdf.hdf_source import HDFSource
from modacor.io.io_sinks import IoSinks
from modacor.io.io_sources import IoSources
from modacor.runner.pipeline import Pipeline

PROJECT_DIR = locate_example_dir("BAM/MOUSE")
sys.path.insert(0, str(PROJECT_DIR))
from mouse_helpers import discover_measurement_pairs


## Configuration

In [ ]:
PIPELINE_PATH = PROJECT_DIR / "pipelines" / "MOUSE_solids.yaml"
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "work" / "supplementary" / "poster_2026" / "figures"
OUTPUT_STEM = "MOUSE_all_configurations_correction_comparison"
SAMPLE_BATCH = 2
MIN_PIXELS_PER_BIN = 20
SYMLOG_LINEAR_THRESHOLD_PERCENT = 1.0
INSET_CONFIGURATIONS = (127, 125, 123, 166)
INSET_ANCHOR_Q_NM = {127: 0.026, 125: 0.075, 123: 0.32, 166: 18.0}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PNG_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}.png"
SVG_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}.svg"
METRICS_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}_metrics.json"


## Process and integrate each configuration

Each pipeline is executed through `IP`. The raw image is captured at `PD_sample` and averaged over any leading acquisition axes. The corrected image, final mask, $q$ map, and pixel-bin index are taken from the completed correction chain. The configured signal weights are unity for this example, so the explicit arithmetic bin means match `IndexedAverager`.

In [ ]:
def process_configuration(pair):
    sources = IoSources()
    sources.register_source(HDFSource(source_reference="sample", resource_location=pair["sample"]))
    sources.register_source(HDFSource(source_reference="background", resource_location=pair["background"]))
    processing_data = ProcessingData()
    sinks = IoSinks()
    scheduler = Pipeline.from_yaml_file(yaml_file=PIPELINE_PATH).create_scheduler()
    scheduler.prepare()

    raw_image = None
    reached_indexing = False
    while scheduler.is_active() and not reached_indexing:
        for node in scheduler.get_ready():
            node.processing_data = processing_data
            node.io_sources = sources
            node.io_sinks = sinks
            node.execute(processing_data)
            step_id = str(node.step_id)
            if step_id == "PD_sample":
                raw_loaded = np.asarray(processing_data["sample"]["signal"].signal, dtype=float)
                acquisition_axes = tuple(range(max(raw_loaded.ndim - 2, 0)))
                raw_image = (
                    np.mean(raw_loaded, axis=acquisition_axes) if acquisition_axes else raw_loaded.copy()
                )
            scheduler.done(node)
            if step_id == "IP":
                reached_indexing = True
                break

    if raw_image is None or not reached_indexing:
        raise RuntimeError(f"Could not obtain indexed data for configuration {pair['configuration']}.")

    sample = processing_data["sample"]
    corrected_image = np.asarray(sample["signal"].signal, dtype=float)
    q_map = np.asarray(sample["Q"].signal, dtype=float)
    final_mask = np.asarray(sample["mask"].signal, dtype=bool)
    pixel_index = np.asarray(sample["pixel_index"].signal, dtype=int)

    scale_valid = (
        ~final_mask
        & np.isfinite(raw_image)
        & np.isfinite(corrected_image)
        & (raw_image > 0)
        & (corrected_image != 0)
    )
    display_scale = float(np.exp(np.median(
        np.log(np.abs(corrected_image[scale_valid]) / raw_image[scale_valid])
    )))

    curve_valid = (
        ~final_mask
        & (pixel_index >= 0)
        & np.isfinite(raw_image)
        & np.isfinite(corrected_image)
        & np.isfinite(q_map)
    )
    indices = pixel_index[curve_valid]
    n_bins = int(indices.max()) + 1
    counts = np.bincount(indices, minlength=n_bins)
    safe_counts = np.maximum(counts, 1)
    q_curve = np.bincount(indices, weights=q_map[curve_valid], minlength=n_bins) / safe_counts
    raw_curve = np.bincount(indices, weights=raw_image[curve_valid], minlength=n_bins) / safe_counts
    corrected_curve = np.bincount(
        indices, weights=corrected_image[curve_valid], minlength=n_bins
    ) / safe_counts
    corrected_mean_per_pixel = corrected_curve[indices]
    corrected_deviation = corrected_image[curve_valid] - corrected_mean_per_pixel
    corrected_sum_squared_deviation = np.bincount(
        indices, weights=corrected_deviation**2, minlength=n_bins
    )
    corrected_sem = np.full(n_bins, np.nan, dtype=float)
    sem_valid = counts > 1
    corrected_sem[sem_valid] = np.sqrt(
        (corrected_sum_squared_deviation[sem_valid] / counts[sem_valid])
        / counts[sem_valid]
    )
    keep = (
        (counts >= MIN_PIXELS_PER_BIN)
        & np.isfinite(q_curve)
        & (raw_curve > 0)
        & (corrected_curve > 0)
    )

    q_plot = q_curve[keep]
    raw_plot = display_scale * raw_curve[keep]
    corrected_plot = corrected_curve[keep]
    effect_percent = 100.0 * (corrected_plot / raw_plot - 1.0)
    return {
        "configuration": int(pair["configuration"]),
        "sample": pair["sample"].name,
        "background": pair["background"].name,
        "display_scale": display_scale,
        "q": q_plot,
        "raw": raw_plot,
        "corrected": corrected_plot,
        "corrected_sem": corrected_sem[keep],
        "effect_percent": effect_percent,
        "corrected_image": corrected_image,
        "final_mask": final_mask,
    }


In [ ]:
pairs = sorted(
    discover_measurement_pairs(DATA_DIR, batch_start=SAMPLE_BATCH, batch_end=SAMPLE_BATCH),
    key=lambda pair: pair["configuration"],
)

previous_logging_disable = logging.root.manager.disable
logging.disable(logging.INFO)
try:
    comparisons = [process_configuration(pair) for pair in pairs]
finally:
    logging.disable(previous_logging_disable)

for comparison in comparisons:
    effect = comparison["effect_percent"]
    print(
        f"{comparison['configuration']}: q={comparison['q'].min():.3g}–"
        f"{comparison['q'].max():.3g} nm^-1; effect={effect.min():+.0f}%–{effect.max():+.0f}%"
    )


## Poster plot

Color identifies the configuration; line style identifies the processing state. The lower panel contains ordinary percentage changes on a symmetric-logarithmic scale that is linear between $-1\%$ and $+1\%$.

In [ ]:
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 14,
    "axes.labelsize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "axes.labelcolor": "#20242b",
    "text.color": "#20242b",
    "axes.edgecolor": "#707782",
    "xtick.color": "#505761",
    "ytick.color": "#505761",
})

figure, (top_axis, effect_axis) = plt.subplots(
    2, 1, figsize=(10.5, 7.4), sharex=True,
    gridspec_kw={"height_ratios": [3, 1], "hspace": 0.06},
    layout="constrained", facecolor="white",
)
configuration_colors = plt.get_cmap("turbo")(np.linspace(0.05, 0.92, len(comparisons)))

for comparison, color in zip(comparisons, configuration_colors):
    top_axis.loglog(
        comparison["q"], comparison["raw"], color=color, linewidth=1.35,
        linestyle=(0, (4, 2)), alpha=0.42, zorder=1,
    )
    top_axis.loglog(
        comparison["q"], comparison["corrected"], color=color,
        linewidth=2.35, alpha=0.96, zorder=2,
    )
    sem_lower = comparison["corrected"] - comparison["corrected_sem"]
    sem_upper = comparison["corrected"] + comparison["corrected_sem"]
    sem_lower = np.where(sem_lower > 0, sem_lower, np.nan)
    top_axis.fill_between(
        comparison["q"], sem_lower, sem_upper, color=color,
        alpha=0.22, linewidth=0, zorder=0,
    )
    effect_axis.semilogx(
        comparison["q"], comparison["effect_percent"],
        color=color, linewidth=2.1, alpha=0.94,
    )

top_axis.set_ylabel(r"Intensity (1/(m sr))")
top_axis.grid(which="major", alpha=0.17)
top_axis.grid(which="minor", alpha=0.05)

effect_axis.set_yscale(
    "symlog", linthresh=SYMLOG_LINEAR_THRESHOLD_PERCENT, linscale=1.0, base=10
)
effect_axis.set_ylim(-120, 100)
effect_axis.axhline(0, color="#69717c", linewidth=1.1)
effect_axis.axhspan(
    -SYMLOG_LINEAR_THRESHOLD_PERCENT, SYMLOG_LINEAR_THRESHOLD_PERCENT,
    color="#dadee5", alpha=0.45, zorder=0,
)
effect_axis.set_xlabel(r"$q$ (nm$^{-1}$)")
effect_axis.set_ylabel("Relative effect (%)")
effect_axis.grid(which="major", alpha=0.17)
effect_axis.grid(which="minor", alpha=0.05)
effect_axis.text(
    0.012, 0.08, "ordinary percent · symlog axis · linear within ±1%",
    transform=effect_axis.transAxes, fontsize=9, color="#5d6470",
)

configuration_handles = [
    Line2D([0], [0], color=color, linewidth=3, label=str(comparison["configuration"]))
    for comparison, color in zip(comparisons, configuration_colors)
]
configuration_legend = top_axis.legend(
    handles=configuration_handles, title="configuration", ncol=5, loc="lower left",
    frameon=False, columnspacing=0.9, handlelength=1.4, fontsize=9, title_fontsize=9,
)
top_axis.add_artist(configuration_legend)
state_handles = [
    Line2D([0], [0], color="#30343b", linewidth=2.5, label="corrected (shading: ±1 SEM)"),
    Line2D(
        [0], [0], color="#737a84", linewidth=1.5, linestyle=(0, (4, 2)), alpha=0.65,
        label="raw × one scale / configuration",
    ),
]
top_axis.legend(handles=state_handles, loc="upper right", frameon=False, fontsize=9)

inset_positions = {
    127: (0.035, 0.52, 0.14, 0.23),
    125: (0.22, 0.64, 0.14, 0.23),
    123: (0.46, 0.65, 0.14, 0.23),
    166: (0.72, 0.65, 0.14, 0.23),
}
plasma = plt.get_cmap("plasma").copy()
plasma.set_bad("#e5e6e9")
for configuration in INSET_CONFIGURATIONS:
    comparison = next(
        item for item in comparisons if item["configuration"] == configuration
    )
    corrected_image = comparison["corrected_image"]
    image_mask = (
        comparison["final_mask"] | ~np.isfinite(corrected_image) | (corrected_image <= 0)
    )
    positive = corrected_image[~image_mask]
    intensity_min, intensity_max = np.quantile(positive, (0.05, 0.995))
    inset_x, inset_y, inset_width, inset_height = inset_positions[configuration]
    inset_axis = top_axis.inset_axes([inset_x, inset_y, inset_width, inset_height])
    inset_axis.imshow(
        np.ma.array(corrected_image, mask=image_mask), origin="lower", cmap=plasma,
        norm=LogNorm(intensity_min, intensity_max), interpolation="nearest",
    )
    inset_axis.set_xticks([])
    inset_axis.set_yticks([])
    inset_axis.set_title(str(configuration), fontsize=9, pad=1, color="#30343b")
    for spine in inset_axis.spines.values():
        spine.set_linewidth(0.7)

    q_anchor = INSET_ANCHOR_Q_NM[configuration]
    anchor_index = int(np.argmin(np.abs(comparison["q"] - q_anchor)))
    top_axis.annotate(
        "",
        xy=(comparison["q"][anchor_index], comparison["corrected"][anchor_index]),
        xycoords="data", xytext=(inset_x + inset_width / 2, inset_y),
        textcoords=top_axis.transAxes,
        arrowprops={
            "arrowstyle": "->", "linewidth": 0.8, "color": "#555d68",
            "shrinkA": 2, "shrinkB": 2, "connectionstyle": "arc3,rad=.08",
        },
        zorder=4,
    )
top_axis.text(
    0.99, 0.52, "2D: log intensity · plasma\nindependent inset scaling",
    transform=top_axis.transAxes, ha="right", va="bottom",
    fontsize=8, color="#5d6470",
)

figure.savefig(PNG_PATH, dpi=300, bbox_inches="tight", facecolor="white")
figure.savefig(SVG_PATH, bbox_inches="tight", facecolor="white")
plt.show()
print(f"PNG: {PNG_PATH}")
print(f"SVG: {SVG_PATH}")


## Export the numerical summary

In [ ]:
report = {
    "sample_batch": SAMPLE_BATCH,
    "effect_definition_percent": "100 * (corrected / (configuration_display_scale * raw) - 1)",
    "display_scale_definition": "median absolute corrected/raw pixel ratio",
    "symlog_linear_threshold_percent": SYMLOG_LINEAR_THRESHOLD_PERCENT,
    "configurations": {},
}
for comparison in comparisons:
    effect = comparison["effect_percent"]
    report["configurations"][str(comparison["configuration"])] = {
        "sample": comparison["sample"],
        "background": comparison["background"],
        "global_display_scale": comparison["display_scale"],
        "q_min_nm^-1": float(comparison["q"].min()),
        "q_max_nm^-1": float(comparison["q"].max()),
        "plotted_bin_count": int(comparison["q"].size),
        "effect_min_percent": float(effect.min()),
        "effect_max_percent": float(effect.max()),
        "median_relative_sem_percent": float(np.nanmedian(
            100.0 * comparison["corrected_sem"] / comparison["corrected"]
        )),
        "effect_quantiles_percent": {
            str(quantile): float(np.quantile(effect, quantile))
            for quantile in (0.05, 0.25, 0.50, 0.75, 0.95)
        },
    }

METRICS_PATH.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print(f"Metrics: {METRICS_PATH}")
